# 01 — Browse the Catalog

This notebook demonstrates how to explore the lab-data catalog using the `labdata.catalog` API.

**Access model:**
- Connects as `labdata_nb` via `LABDATA_READONLY_URL` (falls back to `DATABASE_URL`).
- Queries **only** the `v_*` catalog views — no direct access to production tables.
- Read-only: you cannot modify any catalog data from this notebook.

**What this notebook covers:**
1. List all runs (`catalog.list_runs`).
2. Filter runs by state (e.g. `published`).
3. Browse samples (`catalog.samples`).
4. Browse devices (`catalog.devices`).

In [ ]:
import labdata.catalog as catalog
import pandas as pd

# Show pandas output wider so run_id UUIDs are not truncated
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_columns', 20)

## 1. List all runs

`catalog.list_runs()` queries `v_runs` and returns a DataFrame.
Optional filters: `state`, `sample_id`, `device_id`, `measurement_type`, `limit`.

In [ ]:
df_runs = catalog.list_runs(limit=200)
print(f"Total runs: {len(df_runs)}")
df_runs.head(10)

In [ ]:
# Summary of run states
if not df_runs.empty:
    print("Run state counts:")
    print(df_runs['state'].value_counts().to_string())

## 2. Filter runs by state = 'published'

Pass `state='published'` to restrict results to published runs only.

In [ ]:
df_published = catalog.list_runs(state='published')
print(f"Published runs: {len(df_published)}")
df_published.head()

In [ ]:
# Filter by measurement_type
df_fet = catalog.list_runs(measurement_type='fet_transfer')
print(f"FET transfer runs: {len(df_fet)}")
df_fet[['id', 'sample_id', 'device_id', 'state', 'declared_at']].head()

## 3. Browse samples

`catalog.samples()` returns all rows from `v_samples`.

In [ ]:
df_samples = catalog.samples()
print(f"Samples in catalog: {len(df_samples)}")
df_samples.head()

## 4. Browse devices

`catalog.devices()` returns all rows from `v_devices`.
Pass `sample_id=...` to filter devices for a specific sample.

In [ ]:
df_devices = catalog.devices()
print(f"Devices in catalog: {len(df_devices)}")
df_devices.head()

In [ ]:
# Filter devices for the first sample (if any samples exist)
if not df_samples.empty:
    first_sample_id = df_samples.iloc[0]['id']
    df_sample_devices = catalog.devices(sample_id=first_sample_id)
    print(f"Devices for sample '{first_sample_id}': {len(df_sample_devices)}")
    df_sample_devices.head()

## 5. Inspect a single run

`catalog.get_run(run_id)` returns a dict for a specific run from `v_runs`.
Raises `KeyError` if the run does not exist.

In [ ]:
if not df_runs.empty:
    run_id = df_runs.iloc[0]['id']
    run = catalog.get_run(run_id)
    print(f"Run details for {run_id}:")
    for k, v in run.items():
        print(f"  {k}: {v}")